# From Research to Production — Exporting PyTorch Models

**What you'll learn:**
- Why export matters for deployment
- `torch.export` fundamentals and dynamic shapes
- Graph inspection and transformation
- AOTInductor, NativeRT, and ONNX export paths
- Choosing the right deployment strategy

**Prerequisites:** Basic PyTorch model building (Day 1-3 material)

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch version: {torch.__version__}")
print(f"torch.export available: {hasattr(torch, 'export')}")

## Why Export? Eager vs Graph Mode

PyTorch's default **eager mode** executes operations one at a time — great for debugging, but it carries the Python interpreter overhead everywhere.

**Exporting** captures your model as a static graph (no Python runtime needed), enabling:
- **Deployment** to environments without Python (mobile, embedded, C++ servers)
- **Optimization** — graph-level transformations, operator fusion, quantization
- **Portability** — run the same model across different hardware backends

| Aspect | Eager Mode | Exported Graph |
|--------|-----------|----------------|
| Debugging | Easy (pdb, print) | Hard (static graph) |
| Performance | Python overhead | Optimized |
| Deployment | Needs Python | Standalone |
| Dynamic control flow | Full Python | Limited/structured |

In [ ]:
# A simple model we'll use throughout this notebook
class SimpleClassifier(nn.Module):
    def __init__(self, in_features=784, hidden=256, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleClassifier()
model.eval()

# In eager mode, each op executes immediately
x = torch.randn(1, 784)
output = model(x)
print(f"Eager mode output shape: {output.shape}")
print(f"Eager mode — every op runs through Python interpreter")

## torch.export Basics

`torch.export.export()` traces your model with concrete example inputs and captures a complete graph representation. Unlike `torch.jit.trace` (legacy), `torch.export`:
- Validates that the graph is **sound** (no data-dependent control flow issues)
- Captures the **full graph** including all operations
- Returns an `ExportedProgram` with rich metadata

In [ ]:
# Export the model
example_input = (torch.randn(1, 784),)
exported = torch.export.export(model, example_input)

print(f"Type: {type(exported)}")
print(f"\nExported program structure:")
print(f"  Graph module: {type(exported.graph_module)}")
print(f"  Graph signature: {exported.graph_signature}")
print(f"\nThe exported graph (first 30 lines):")
graph_str = str(exported.graph_module.graph)
for line in graph_str.split('\n')[:30]:
    print(f"  {line}")

## Verifying Exported Programs

The exported program is callable — you can run it like the original model and verify the outputs match:

In [ ]:
# Run the exported program and compare with eager
test_input = torch.randn(4, 784)

eager_output = model(test_input)
exported_output = exported.module()(test_input)

print(f"Eager output shape: {eager_output.shape}")
print(f"Export output shape: {exported_output.shape}")
print(f"Outputs match: {torch.allclose(eager_output, exported_output, atol=1e-6)}")
print(f"Max difference: {(eager_output - exported_output).abs().max():.2e}")

## Dynamic Shapes with the Dim API

By default, `torch.export` fixes all tensor shapes. But real deployments need flexibility — different batch sizes, variable sequence lengths. The `Dim` API lets you specify which dimensions are dynamic.

In [ ]:
from torch.export import Dim

# Define dynamic dimensions
batch = Dim("batch", min=1, max=128)

# Export with dynamic batch dimension
dynamic_exported = torch.export.export(
    model,
    (torch.randn(1, 784),),
    dynamic_shapes={"x": {0: batch}},
)

# Now it works with any batch size!
for bs in [1, 4, 16, 64]:
    test_x = torch.randn(bs, 784)
    out = dynamic_exported.module()(test_x)
    print(f"Batch size {bs:3d} → output shape: {out.shape}")

### Multiple Dynamic Dimensions

For models like sequence models, you may need both batch and sequence length to be dynamic:

In [ ]:
# A sequence model with variable batch and sequence length
class SeqModel(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, nhead=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        x = self.embed(tokens)
        x = self.encoder(x)
        return self.head(x)

seq_model = SeqModel()
seq_model.eval()

batch = Dim("batch", min=1, max=64)
seq_len = Dim("seq_len", min=1, max=512)

exported_seq = torch.export.export(
    seq_model,
    (torch.randint(0, 1000, (2, 16)),),
    dynamic_shapes={"tokens": {0: batch, 1: seq_len}},
)

# Test with various shapes
for bs, sl in [(1, 8), (4, 32), (8, 128)]:
    tokens = torch.randint(0, 1000, (bs, sl))
    out = exported_seq.module()(tokens)
    print(f"Batch={bs}, SeqLen={sl} → output: {out.shape}")

## Saving and Loading Exported Programs

Exported programs can be serialized to `.pt2` format for later use:

In [ ]:
import tempfile, os

# Save the exported program
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "classifier.pt2")
    torch.export.save(dynamic_exported, path)
    file_size = os.path.getsize(path)
    print(f"Saved to: {path}")
    print(f"File size: {file_size / 1024:.1f} KB")

    # Load it back
    loaded = torch.export.load(path)
    print(f"\nLoaded type: {type(loaded)}")

    # Verify it still works
    test_input = torch.randn(8, 784)
    original_out = dynamic_exported.module()(test_input)
    loaded_out = loaded.module()(test_input)
    print(f"Outputs match after save/load: {torch.allclose(original_out, loaded_out)}")

## Graph Inspection

One of the most powerful aspects of exported programs is the ability to inspect and manipulate the computation graph:

In [ ]:
# Inspect the graph: list all operations
graph = exported.graph_module.graph

print("=== All operations in the exported graph ===")
op_counts = {}
for node in graph.nodes:
    op_counts[node.op] = op_counts.get(node.op, 0) + 1
    if node.op == "call_function":
        print(f"  {node.name}: {node.target}")

print(f"\n=== Node type summary ===")
for op_type, count in sorted(op_counts.items()):
    print(f"  {op_type}: {count}")

In [ ]:
# Print the full graph with node details
print("=== Detailed graph nodes ===")
for node in graph.nodes:
    meta = node.meta.get("val", None)
    shape_info = ""
    if hasattr(meta, "shape"):
        shape_info = f" → {meta.shape}"
    print(f"  [{node.op}] {node.name}{shape_info}")

## AOTInductor Overview

**AOTInductor** (Ahead-of-Time Inductor) compiles your model to a standalone shared library (`.so` file) that can run without Python. This is the recommended path for server-side deployment.

```
Python model → torch.export → AOTInductor → .so library → C++ runtime
```

Key benefits:
- No Python interpreter needed at inference time
- Hardware-specific optimizations (fused kernels, memory planning)
- Can be loaded in C++ applications via NativeRT

In [ ]:
# AOTInductor workflow (conceptual — requires GPU for full compilation)
print("AOTInductor workflow:")
print("=" * 50)
print("""
# Step 1: Export the model
exported = torch.export.export(model, example_inputs)

# Step 2: Compile to a package (.pt2)
# This creates an optimized, self-contained artifact
torch._inductor.aoti_compile_and_package(
    exported,
    package_path="model.pt2",
)

# Step 3: Load and run (no Python needed at runtime)
compiled = torch._inductor.aoti_load_package("model.pt2")
output = compiled(input_tensor)
""")

print("In C++ (using NativeRT):")
print("=" * 50)
print("""
// Load the .so library
auto model = torch::inductor::AOTIModelContainerRunnerCpu("model.so");

// Run inference
auto outputs = model.run({input_tensor});
""")

## NativeRT Overview

**NativeRT** is the C++ inference engine that loads AOTInductor artifacts. It provides:
- Zero Python dependency at runtime
- Minimal binary size
- Thread-safe inference
- Support for dynamic shapes

```
Training (Python)        Deployment (C++)
─────────────────       ─────────────────
model.py                model_server.cpp
  │                       │
  ├─ torch.export        ├─ Load .so
  ├─ AOTInductor         ├─ Create runner
  └─ Output: model.so   └─ Run inference
```

## ONNX Export

ONNX (Open Neural Network Exchange) is an open format for representing ML models. It enables interoperability between frameworks and deployment to specialized runtimes (ONNX Runtime, TensorRT, etc.).

In [ ]:
# ONNX export with dynamo backend
import tempfile

simple_model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
simple_model.eval()

example = torch.randn(1, 32)

with tempfile.TemporaryDirectory() as tmpdir:
    onnx_path = os.path.join(tmpdir, "model.onnx")

    # Export using dynamo-based ONNX export
    onnx_program = torch.onnx.export(
        simple_model,
        (example,),
        dynamo=True,
    )
    onnx_program.save(onnx_path)

    file_size = os.path.getsize(onnx_path)
    print(f"ONNX model saved: {file_size / 1024:.1f} KB")
    print(f"\nONNX export completed successfully!")
    print(f"The model can now be loaded with onnxruntime or other ONNX runtimes")

## Comparison: Which Deployment Path?

| Path | Python Needed? | Target | Dynamic Shapes | Best For |
|------|---------------|--------|----------------|----------|
| **torch.export** | Yes (PyTorch) | Python apps | Yes (Dim API) | Python microservices |
| **AOTInductor** | No | Server (C++) | Yes | High-perf C++ servers |
| **NativeRT** | No | C++ apps | Yes | Embedded C++ inference |
| **ONNX** | No | Any runtime | Yes | Cross-framework, edge |
| **ExecuTorch** | No | Mobile/edge | Limited | Phone/IoT devices |

**Decision guide:**
- **Staying in Python?** → `torch.export` + `torch.compile`
- **C++ server?** → AOTInductor + NativeRT
- **Need portability?** → ONNX
- **Mobile/Edge?** → ExecuTorch

## 🏋️ Try It Yourself: Export with Dynamic Shapes

**Exercise:** Create a convolutional model and export it with two dynamic dimensions:
1. Dynamic batch size
2. Dynamic spatial size (height/width)

Verify the exported model works with 3 different input shapes.

In [ ]:
# Exercise starter code
class ConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(32, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

conv_model = ConvModel()
conv_model.eval()

# Your code here:
# 1. Define batch and spatial Dim objects
# 2. Export with dynamic_shapes
# 3. Test with different shapes

# Solution:
batch_dim = Dim("batch", min=1, max=64)
height_dim = Dim("height", min=8, max=256)
width_dim = Dim("width", min=8, max=256)

exported_conv = torch.export.export(
    conv_model,
    (torch.randn(1, 3, 32, 32),),
    dynamic_shapes={"x": {0: batch_dim, 2: height_dim, 3: width_dim}},
)

for shape in [(1, 3, 16, 16), (8, 3, 64, 64), (4, 3, 128, 128)]:
    test = torch.randn(*shape)
    out = exported_conv.module()(test)
    print(f"Input {shape} → Output {out.shape} ✓")

## Key Takeaways

1. **torch.export** captures your model as a complete, validated graph — the foundation for all deployment paths

2. **Dynamic shapes** via the `Dim` API let exported models handle variable batch sizes, sequence lengths, and spatial dimensions

3. **Exported programs** can be saved (`.pt2`) and loaded, making them portable across machines

4. **Graph inspection** lets you list all ops, examine shapes, and understand what the model computes

5. **AOTInductor** compiles to standalone `.so` libraries — no Python needed at inference, maximum performance

6. **NativeRT** provides the C++ runtime for loading AOTInductor artifacts

7. **ONNX export** (via `torch.onnx.export` with `dynamo=True`) enables cross-framework deployment

8. **Choose your path:** Python deployment → torch.export, C++ server → AOTInductor, portability → ONNX, mobile → ExecuTorch